# Eixo 4: Impacto Socioambiental

## Fontes de Dados
- **Embargos (IBAMA)**: Lista de propriedades punidas por crimes ambientais
- **Atlas Brasil (IPEA/PNUD)**: IDHM, renda e longevidade municipal

---

## Análise 1: Impacto dos Embargos na Produção Agropecuária

### Especificação BDD

**Feature**: Avaliar se embargos do IBAMA reduzem a produção agropecuária local

**Scenario**: Comparar produção antes e depois de picos de embargos

- **GIVEN** que tenho dados de embargos IBAMA por município e ano
- **AND** que tenho dados de produção agropecuária (PAM/PPM) por município e ano
- **WHEN** eu identifico municípios com picos de embargos (aumento significativo em um ano)
- **AND** comparo a produção nos 3 anos antes do pico com os 3 anos depois
- **AND** calculo a variação percentual média da produção
- **AND** aplico um modelo de diferenças-em-diferenças com grupo controle
- **THEN** devo obter o efeito médio dos embargos na produção
- **AND** devo identificar se a produção diminuiu, permaneceu estável ou aumentou
- **AND** devo calcular o intervalo de confiança do efeito
- **AND** devo testar se o efeito é estatisticamente significativo

### Matemática em Linguagem de Negócio

**1. Identificação de Picos de Embargos:**
- Pico = aumento > 2 desvios-padrão em relação à média histórica do município
- Isso identifica anos com atividade de fiscalização anormalmente alta
- Evita anos normais que não representam mudança de política

**2. Variação da Produção (Pré vs Pós):**
- Média pré = média da produção nos 3 anos antes do pico
- Média pós = média da produção nos 3 anos depois do pico
- Variação % = (Média pós - Média pré) / Média pré × 100%
- Variação positiva = produção aumentou após embargos
- Variação negativa = produção diminuiu após embargos

**3. Diferenças-em-Diferenças (DiD):**
- Compara municípios com embargos (tratamento) vs sem embargos (controle)
- Efeito DiD = (Tratamento pós - Tratamento pré) - (Controle pós - Controle pré)
- Controla por tendências pré-existentes em ambos os grupos
- Isola o efeito causal dos embargos

**4. Intervalo de Confiança:**
- Calcula a margem de erro ao redor da estimativa
- IC 95% = estimativa ± 1.96 × erro padrão
- Se o IC não inclui zero, o efeito é estatisticamente significativo
- Se o IC inclui zero, não podemos rejeitar a hipótese nula (efeito pode ser zero)

**5. Interpretação de Negócio:**
- Efeito negativo significativo: embargos reduzem produção (política eficaz)
- Efeito não significativo: embargos não afetam produção (produtores adaptam)
- Efeito positivo significativo: produção aumenta após embargos (paradoxo)
- Possíveis explicações para efeito não significativo:
  - Produtores desviam produção para municípios vizinhos (spillover)
  - Produtores mudam para culturas não monitoradas
  - Produção informal aumenta (não declarada)

**6. Limitações:**
- Endogeneidade: embargos podem ser aplicados em municípios já em declínio
- Viés de medição: produção declarada pode não refletir produção real
- Não considera adaptação: produtores podem migrar para outras atividades

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

# Configurações
DATA_DIR = Path("/app/data")

# Carregar dados de embargos
embargos_path = DATA_DIR / "02_silver/embargos_por_municipio_ano.parquet"
df_embargos = pd.read_parquet(embargos_path)

print("Dados de Embargos carregados:")
print(f"Shape: {df_embargos.shape}")
print(f"\nColunas: {df_embargos.columns.tolist()}")
print(f"\nPrimeiras linhas:")
df_embargos.head()

In [ ]:
# Carregar dados de produção (PAM e PPM)
pam_path = DATA_DIR / "02_silver/pam_consolidado.parquet"
df_pam = pd.read_parquet(pam_path)

ppm_path = DATA_DIR / "02_silver/ppm_consolidado.parquet"
df_ppm = pd.read_parquet(ppm_path)

# Criar proxy de produção total (valor produção PAM + proxy PPM)
# Nota: PPM não tem valor direto, usar rebanho como proxy
df_producao = df_pam[['codigo_ibge', 'ano', 'valor_producao']].copy()

print("Dados de Produção carregados:")
print(f"PAM: {df_pam.shape}")
print(f"PPM: {df_ppm.shape}")

In [ ]:
# Identificar picos de embargos por município
# Pico = aumento > 2 desvios-padrão em relação à média histórica

def identificar_picos_embargos(df, threshold_std=2):
    """
    Identifica anos com picos de embargos para cada município.
    
    Args:
        df: DataFrame com embargos por município e ano
        threshold_std: número de desvios-padrão para definir pico
    
    Returns:
        DataFrame com anos de pico por município
    """
    picos = []
    
    for municipio in df['codigo_ibge'].unique():
        df_mun = df[df['codigo_ibge'] == municipio].sort_values('ano')
        
        if len(df_mun) < 3:  # Precisa de pelo menos 3 anos para calcular média
            continue
        
        media = df_mun['num_embargos'].mean()
        std = df_mun['num_embargos'].std()
        
        if std == 0:  # Evitar divisão por zero
            continue
        
        # Identificar anos acima do threshold
        anos_pico = df_mun[
            df_mun['num_embargos'] > (media + threshold_std * std)
        ]['ano'].tolist()
        
        for ano_pico in anos_pico:
            picos.append({
                'codigo_ibge': municipio,
                'ano_pico': ano_pico,
                'num_embargos_pico': df_mun[df_mun['ano'] == ano_pico]['num_embargos'].values[0],
                'media_historica': media,
                'std_historico': std
            })
    
    return pd.DataFrame(picos)

# Identificar picos
df_picos = identificar_picos_embargos(df_embargos, threshold_std=2)

print(f"Identificados {len(df_picos)} picos de embargos em {df_picos['codigo_ibge'].nunique()} municípios")
df_picos.head()

In [ ]:
# Para cada pico, calcular produção antes e depois
resultados_impacto = []

for _, row in df_picos.iterrows():
    municipio = row['codigo_ibge']
    ano_pico = row['ano_pico']
    
    # Produção nos 3 anos antes
    anos_pre = list(range(ano_pico - 3, ano_pico))
    producao_pre = df_producao[
        (df_producao['codigo_ibge'] == municipio) & 
        (df_producao['ano'].isin(anos_pre))
    ]['valor_producao'].mean()
    
    # Produção nos 3 anos depois
    anos_pos = list(range(ano_pico + 1, ano_pico + 4))
    producao_pos = df_producao[
        (df_producao['codigo_ibge'] == municipio) & 
        (df_producao['ano'].isin(anos_pos))
    ]['valor_producao'].mean()
    
    # Calcular variação
    if producao_pre > 0 and not pd.isna(producao_pos):
        variacao_pct = ((producao_pos - producao_pre) / producao_pre) * 100
    else:
        variacao_pct = np.nan
    
    resultados_impacto.append({
        'codigo_ibge': municipio,
        'ano_pico': ano_pico,
        'producao_pre': producao_pre,
        'producao_pos': producao_pos,
        'variacao_pct': variacao_pct
    })

df_impacto = pd.DataFrame(resultados_impacto)
df_impacto = df_impacto.dropna()

print("Impacto dos Embargos na Produção:")
df_impacto.head()

In [ ]:
# Estatísticas descritivas da variação
print("Estatísticas da Variação da Produção após Embargos:")
print(df_impacto['variacao_pct'].describe())

# Teste t para verificar se a variação média é diferente de zero
t_stat, p_value = stats.ttest_1samp(df_impacto['variacao_pct'], 0)

print(f"\nTeste t (variação vs 0):")
print(f"Estatística t: {t_stat:.4f}")
print(f"Valor-p: {p_value:.4f}")

if p_value < 0.05:
    print("\nA variação é estatisticamente significativa (p < 0.05)")
else:
    print("\nA variação NÃO é estatisticamente significativa (p >= 0.05)")

In [ ]:
# Visualizar distribuição da variação
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(df_impacto['variacao_pct'], bins=30, edgecolor='black', alpha=0.7)
plt.axvline(x=0, color='red', linestyle='--', label='Sem variação')
plt.axvline(x=df_impacto['variacao_pct'].mean(), color='green', linestyle='--', label=f'Média: {df_impacto["variacao_pct"].mean():.1f}%')
plt.xlabel('Variação da Produção (%)', fontsize=12)
plt.ylabel('Frequência', fontsize=12)
plt.title('Distribuição da Variação da Produção', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
sns.boxplot(y=df_impacto['variacao_pct'])
plt.axhline(y=0, color='red', linestyle='--', alpha=0.5)
plt.ylabel('Variação da Produção (%)', fontsize=12)
plt.title('Boxplot da Variação', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

### Conclusão da Análise 1

**Interpretação de Negócio:**
- A variação média da produção após picos de embargos é [X.XX]%
- O valor-p de [X.XXXX] indica que o efeito é [significativo/não significativo]
- Isso sugere que [embargos reduzem/não afetam/aumentam] a produção local

**Possíveis Explicações:**
- Se variação não significativa: produtores podem estar desviando produção para municípios vizinhos
- Se variação negativa: embargos estão sendo eficazes na contenção
- Se variação positiva: pode haver adaptação ou mudança para atividades não monitoradas

**Limitações:**
- Não implementa grupo controle (diferenças-em-diferenças completo)
- Pode haver endogeneidade (embargos aplicados em municípios já em declínio)
- Produção declarada pode não refletir produção real

---
## Análise 2: Correlação entre Desmatamento e IDHM

### Especificação BDD

**Feature**: Avaliar se desmatamento gera desenvolvimento social (paradoxo do desenvolvimento)

**Scenario**: Correlacionar crescimento do desmatamento com variação do IDHM

- **GIVEN** que tenho dados de desmatamento (PRODES) por município
- **AND** que tenho dados de IDHM (Atlas Brasil) por município
- **AND** que tenho dados de PIB Agropecuário por município
- **WHEN** eu calculo o delta de desmatamento entre dois períodos
- **AND** calculo o delta de IDHM no mesmo período
- **AND** calculo o delta de PIB Agropecuário no mesmo período
- **AND** calculo a correlação de Pearson entre delta desmatamento e delta IDHM
- **AND** calculo a correlação entre delta PIB e delta IDHM
- **AND** crio um scatter plot com tamanho da bolha = delta desmatamento
- **THEN** devo obter o coeficiente de correlação entre desmatamento e IDHM
- **AND** devo identificar se desmatamento correlaciona-se com desenvolvimento social
- **AND** devo verificar se PIB correlaciona-se melhor com IDHM que desmatamento

### Matemática em Linguagem de Negócio

**1. Delta de IDHM:**
- Delta IDHM = IDHM_final - IDHM_inicial
- IDHM varia de 0 a 1 (quanto maior, melhor)
- Delta positivo = melhoria no desenvolvimento humano
- Delta negativo = piora no desenvolvimento humano
- Delta de 0.01 a 0.05 = melhoria moderada
- Delta > 0.05 = melhoria significativa

**2. Correlação de Pearson:**
- Mede relação linear entre duas variáveis
- Varia de -1 a 1:
  - r ≈ 1: correlação positiva forte (desmatamento alto → IDHM alto)
  - r ≈ 0: sem correlação
  - r ≈ -1: correlação negativa forte (desmatamento alto → IDHM baixo)
- r > 0.3: correlação moderada/forte
- r < 0.3: correlação fraca

**3. Interpretação de Negócio:**
- Correlação positiva forte (r > 0.5): desmatamento associado a melhor IDHM
  - Pode indicar que desmatamento gera desenvolvimento (paradoxo)
  - Ou pode haver variável omitida (ex: investimento em infraestrutura)
- Correlação fraca (r < 0.3): desmatamento não associado a IDHM
  - Desmatamento não gera desenvolvimento social
  - Riqueza pode estar concentrada (não distribuída)
- Correlação PIB-IDHM > correlação Desmatamento-IDHM:
  - PIB é melhor preditor de desenvolvimento que desmatamento
  - Desmatamento pode não ser necessário para desenvolvimento

**4. Limitações:**
- Correlação não implica causalidade
- Pode haver variáveis omitidas (infraestrutura, educação, saúde)
- IDHM é agregado, não captura distribuição de renda
- Benefícios de desmatamento podem ser de longo prazo (gerações)
- Custos ambientais não são capturados pelo IDHM

In [ ]:
# Carregar dados IDHM
idhm_path = DATA_DIR / "02_silver/idhm_municipal_interpolado.parquet"
df_idhm = pd.read_parquet(idhm_path)

print("Dados IDHM carregados:")
print(f"Shape: {df_idhm.shape}")
print(f"\nColunas: {df_idhm.columns.tolist()}")
print(f"\nPrimeiras linhas:")
df_idhm.head()

In [ ]:
# Definir período de análise
ano_inicial_idhm = 2010
ano_final_idhm = 2020

# Calcular delta de IDHM por município
idhm_inicial = df_idhm[df_idhm['ano'] == ano_inicial_idhm].groupby('codigo_ibge')['idhm'].mean()
idhm_final = df_idhm[df_idhm['ano'] == ano_final_idhm].groupby('codigo_ibge')['idhm'].mean()

delta_idhm = (idhm_final - idhm_inicial).reset_index()
delta_idhm.columns = ['codigo_ibge', 'delta_idhm']

# Calcular delta de PIB Agropecuário (já calculado anteriormente)
# Usar delta_vab do cálculo do ICA

# Merge deltas
df_correlacao = pd.merge(delta_desmatamento, delta_idhm, on='codigo_ibge', how='inner')
df_correlacao = pd.merge(df_correlacao, delta_vab, on='codigo_ibge', how='inner')

# Remover valores nulos
df_correlacao = df_correlacao.dropna()

print("Dados para Correlação:")
df_correlacao.head()

In [ ]:
# Calcular correlações
corr_desmatamento_idhm = df_correlacao['delta_desmatamento_ha'].corr(df_correlacao['delta_idhm'])
corr_pib_idhm = df_correlacao['delta_vab_agro'].corr(df_correlacao['delta_idhm'])
corr_desmatamento_pib = df_correlacao['delta_desmatamento_ha'].corr(df_correlacao['delta_vab_agro'])

print("Correlações:")
print(f"Desmatamento vs IDHM: r = {corr_desmatamento_idhm:.4f}")
print(f"PIB Agro vs IDHM: r = {corr_pib_idhm:.4f}")
print(f"Desmatamento vs PIB Agro: r = {corr_desmatamento_pib:.4f}")

# Testar significância estatística
n = len(df_correlacao)
t_desmatamento_idhm = corr_desmatamento_idhm * np.sqrt((n-2) / (1 - corr_desmatamento_idhm**2))
p_desmatamento_idhm = 2 * (1 - stats.t.cdf(abs(t_desmatamento_idhm), n-2))

t_pib_idhm = corr_pib_idhm * np.sqrt((n-2) / (1 - corr_pib_idhm**2))
p_pib_idhm = 2 * (1 - stats.t.cdf(abs(t_pib_idhm), n-2))

print(f"\nTestes de significância:")
print(f"Desmatamento vs IDHM: p = {p_desmatamento_idhm:.4f}")
print(f"PIB Agro vs IDHM: p = {p_pib_idhm:.4f}")

In [ ]:
# Scatter plot: Delta PIB vs Delta IDHM (tamanho = delta desmatamento)
plt.figure(figsize=(10, 8))

scatter = plt.scatter(
    df_correlacao['delta_vab_agro'],
    df_correlacao['delta_idhm'],
    s=df_correlacao['delta_desmatamento_ha'] / 100,  # Escalonar tamanho
    c=df_correlacao['delta_desmatamento_ha'],
    alpha=0.5,
    cmap='YlOrRd',
    edgecolors='black',
    linewidths=0.5
)

plt.colorbar(scatter, label='Delta Desmatamento (ha)')
plt.xlabel('Delta PIB Agropecuário (R$)', fontsize=12)
plt.ylabel('Delta IDHM', fontsize=12)
plt.title('Paradoxo do Desenvolvimento: PIB vs IDHM (tamanho = Desmatamento)', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

# Adicionar linha de tendência PIB-IDHM
z = np.polyfit(df_correlacao['delta_vab_agro'], df_correlacao['delta_idhm'], 1)
p = np.poly1d(z)
plt.plot(df_correlacao['delta_vab_agro'], 
         p(df_correlacao['delta_vab_agro']), 
         "b--", alpha=0.8, linewidth=2, 
         label=f'Tendência PIB-IDHM (r={corr_pib_idhm:.3f})')

plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

### Conclusão da Análise 2

**Interpretação de Negócio:**
- A correlação entre desmatamento e IDHM é [r = valor], [FORTE/MODERADA/FRACA]
- A correlação entre PIB Agropecuário e IDHM é [r = valor], [FORTE/MODERADA/FRACA]
- Isso sugere que [PIB/desmatamento] é melhor preditor de desenvolvimento social

**Paradoxo do Desenvolvimento:**
- Se correlação desmatamento-IDHM é baixa: desmatamento não gera desenvolvimento
- Se correlação PIB-IDHM é alta: crescimento econômico gera desenvolvimento
- Isso indica que [é possível/é difícil] ter desenvolvimento sem desmatamento

**Limitações:**
- Correlação não implica causalidade
- Variáveis omitidas podem explicar resultados
- IDHM não captura distribuição de renda
- Custos ambientais não são considerados

---
## Resumo do Eixo 4: Impacto Socioambiental

**Principais Descobertas:**
1. Impacto dos embargos na produção: variação média de [X.XX]%
2. Correlação desmatamento-IDHM: r = [valor]
3. Correlação PIB-IDHM: r = [valor]

**Implicações de Negócio:**
- [Insights sobre eficácia de fiscalização]
- [Insights sobre paradoxo do desenvolvimento]

**Limitações Metodológicas:**
- [Listar limitações identificadas]

**Próximos Passos:**
- [Implementar DiD completo para embargos]
- [Integrar com dados de distribuição de renda]
- [Calcular custos externos ambientais]